# logsumexp-cross-entropy — worked example 2: logsumexp CE survives where naive softmax overflows

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `logsumexp-cross-entropy`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

The naive cross-entropy `-log(softmax(logits)[target])` computes `exp(logits)` directly, which overflows to inf in float32 once any logit exceeds ~88. The logsumexp form gives the same answer for moderate logits but stays finite for extreme ones.

## Worked solution

We contrast the two formulations on the same large logits.

1. **Stable form.** `logsumexp(logits) - picked` as before, in float32.
2. **Naive form.** `probs = exp(logits) / exp(logits).sum()` then `-log(probs[target])`. For a logit of 100, `exp(100)` is already > float32 max, so `probs` becomes `nan`/`inf` and the naive loss is non-finite.
3. **Agreement on moderate logits.** For small logits both agree to tolerance — proof the stable form is not changing the math, only the numerics.

The demo prints that the stable loss is finite while the naive loss is not, on a logit of 100.

In [ ]:
import torch as t

t.manual_seed(1)

def ce_stable(logits, target):
    lse = t.logsumexp(logits, dim=-1)
    B = logits.shape[0]
    return (lse - logits[t.arange(B), target]).mean()

def ce_naive(logits, target):
    e = t.exp(logits)
    probs = e / e.sum(dim=-1, keepdim=True)
    B = logits.shape[0]
    return -t.log(probs[t.arange(B), target]).mean()

big = t.tensor([[100.0, 0.0, 0.0]], dtype=t.float32)
tgt = t.tensor([0])
print('stable finite:', t.isfinite(ce_stable(big, tgt)).item())
print('naive  finite:', t.isfinite(ce_naive(big, tgt)).item())

small = t.tensor([[1.0, 2.0, 0.5]])
print('agree on moderate logits:',
      t.allclose(ce_stable(small, tgt), ce_naive(small, tgt), atol=1e-5))